# 05 - Custom Models Evaluation (Model A vs. Model B)

Bring together `02_model_A_standard_cnn.ipynb`'s and `03_model_B_lightweight_cnn.ipynb`'s saved results to build the required comparison table, and discuss the standard-vs-depthwise-separable trade-offs.

**Expects** `results/metrics/model_a.json` and `results/metrics/model_b.json` to already exist (i.e. both notebooks 02 and 03 have been run), each with the schema: `model_name`, `dataset_name`, `test_accuracy`, `test_precision_macro`, `test_recall_macro`, `confusion_matrix`, `class_names`, `total_params`, `model_size_kb`, `epochs`, `avg_time_per_epoch_sec`, `history`.

In [ ]:
# --- Environment setup (Google Colab + shared Google Drive, with a local fallback) ---
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the team's shared Drive - this is where 02/03 saved their metrics.
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/EN3150-A03-Resource-Constrained-CNN"
else:
    # Not in Colab (e.g. running locally to sanity-check the code) - use
    # this repo's own local folders instead of Drive.
    REPO_DIR = os.path.abspath("..")

METRICS_DIR = os.path.join(REPO_DIR, "results", "metrics")
TABLES_DIR = os.path.join(REPO_DIR, "results", "tables")
FIGURES_FINAL_DIR = os.path.join(REPO_DIR, "results", "figures", "final")

for d in (TABLES_DIR, FIGURES_FINAL_DIR):
    os.makedirs(d, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("REPO_DIR:", REPO_DIR)


## 1. Imports

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Load Saved Metrics

In [ ]:
def load_metrics(name: str) -> dict:
    with open(os.path.join(METRICS_DIR, f"{name}.json")) as f:
        return json.load(f)

metrics_a = load_metrics("model_a")
metrics_b = load_metrics("model_b")

print("Loaded:", metrics_a["model_name"], "and", metrics_b["model_name"])


## 3. Comparison Table

Required columns: total parameter count, estimated model size on disk (KB), training time per epoch, and test accuracy.

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": metrics_a["model_name"],
        "Total Parameters": metrics_a["total_params"],
        "Model Size (KB)": metrics_a["model_size_kb"],
        "Avg Time/Epoch (s)": metrics_a["avg_time_per_epoch_sec"],
        "Test Accuracy": metrics_a["test_accuracy"],
    },
    {
        "Model": metrics_b["model_name"],
        "Total Parameters": metrics_b["total_params"],
        "Model Size (KB)": metrics_b["model_size_kb"],
        "Avg Time/Epoch (s)": metrics_b["avg_time_per_epoch_sec"],
        "Test Accuracy": metrics_b["test_accuracy"],
    },
])

tables_path = os.path.join(TABLES_DIR, "custom_models.csv")
comparison_df.to_csv(tables_path, index=False)
print("Saved:", tables_path)
comparison_df


## 4. Side-by-Side Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metrics in zip(axes, (metrics_a, metrics_b)):
    cm = np.array(metrics["confusion_matrix"])
    class_names = metrics["class_names"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(metrics["model_name"])

plt.tight_layout()
cm_side_by_side_path = os.path.join(FIGURES_FINAL_DIR, "confusion_matrices_model_a_vs_b.png")
plt.savefig(cm_side_by_side_path, dpi=150)
plt.show()
print("Saved:", cm_side_by_side_path)


## 5. Precision / Recall Comparison

In [ ]:
metric_labels = ["Test Accuracy", "Precision (macro)", "Recall (macro)"]
model_a_values = [metrics_a["test_accuracy"], metrics_a["test_precision_macro"], metrics_a["test_recall_macro"]]
model_b_values = [metrics_b["test_accuracy"], metrics_b["test_precision_macro"], metrics_b["test_recall_macro"]]

x = np.arange(len(metric_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width / 2, model_a_values, width, label=metrics_a["model_name"])
ax.bar(x + width / 2, model_b_values, width, label=metrics_b["model_name"])
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model A vs. Model B - Test Metrics")
ax.legend()
plt.tight_layout()

metrics_comparison_path = os.path.join(FIGURES_FINAL_DIR, "model_a_vs_b_metrics.png")
plt.savefig(metrics_comparison_path, dpi=150)
plt.show()
print("Saved:", metrics_comparison_path)


## 6. Discussion: Standard vs. Depthwise Separable Convolutions

TODO (fill in after running notebooks 02, 03, and this notebook, using the table/plots above): discuss the trade-offs observed when moving from standard convolutions (Model A) to depthwise separable convolutions (Model B) - e.g. how much smaller/faster Model B is, and at what cost (if any) to accuracy. Note whether Model B still meets the assignment's <=100,000 parameter constraint (see `03_model_B_lightweight_cnn.ipynb`, Section 7).

## 7. Save Results

In [ ]:
print("Saved artifacts:")
print(" -", tables_path)
print(" -", cm_side_by_side_path)
print(" -", metrics_comparison_path)


## 8. Conclusion

In [ ]:
name_a, name_b = metrics_a["model_name"], metrics_b["model_name"]
params_a, params_b = metrics_a["total_params"], metrics_b["total_params"]
size_a, size_b = metrics_a["model_size_kb"], metrics_b["model_size_kb"]
time_a, time_b = metrics_a["avg_time_per_epoch_sec"], metrics_b["avg_time_per_epoch_sec"]
acc_a, acc_b = metrics_a["test_accuracy"], metrics_b["test_accuracy"]

param_ratio = params_a / params_b
size_ratio = size_a / size_b
accuracy_diff = acc_a - acc_b

print(
    f"{name_a} vs. {name_b}:\n"
    f"  Parameters:      {params_a:,} vs. {params_b:,}  ({param_ratio:.1f}x)\n"
    f"  Model size:      {size_a:.1f} KB vs. {size_b:.1f} KB  ({size_ratio:.1f}x)\n"
    f"  Avg time/epoch:  {time_a:.1f}s vs. {time_b:.1f}s\n"
    f"  Test accuracy:   {acc_a:.4f} vs. {acc_b:.4f}  (diff: {accuracy_diff:+.4f})"
)


TODO: summarize which model is preferable and under what constraints (e.g. Model B for strict edge/memory budgets even at a small accuracy cost, or Model A if a bit more memory/compute is available and accuracy matters more). Carry these numbers into `08_final_comparison.ipynb`.